### Imports and paths

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed"); PROC_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR  = Path("../figures");        FIG_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 30)

### Sensor names

In [2]:
sensor_map = {
    "s1": "T2_fan_inlet_temp", "s2": "T24_LPC_outlet_temp", "s3": "T30_HPC_outlet_temp",
    "s4": "T50_LPT_outlet_temp", "s5": "P2_fan_inlet_pressure", "s6": "P15_bypass_duct_pressure",
    "s7": "P30_HPC_outlet_pressure", "s8": "Nf_fan_speed", "s9": "Nc_core_speed",
    "s10": "epr_engine_pressure_ratio", "s11": "Ps30_HPC_static_pressure",
    "s12": "phi_fuel_flow_to_Ps30", "s13": "NRf_corrected_fan_speed",
    "s14": "NRc_corrected_core_speed", "s15": "BPR_bypass_ratio",
    "s16": "farB_burner_fuel_air_ratio", "s17": "htBleed_bleed_enthalpy",
    "s18": "Nf_dmd_demanded_fan_speed", "s19": "PCNfR_dmd_demanded_corr_fan_speed",
    "s20": "W31_HPT_coolant_bleed", "s21": "W32_LPT_coolant_bleed",
}
cols = ["unit", "cycle", "op1", "op2", "op3"] + list(sensor_map)

### Load the data and add RUL

In [3]:
df = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=cols)

RUL_CAP = 125
df["RUL"]     = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
df["RUL_cap"] = df["RUL"].clip(upper=RUL_CAP)

print("Shape:", df.shape, "| engines:", df["unit"].nunique())

Shape: (20631, 28) | engines: 100


### Apply sensor selection rule

In [4]:
sensors = list(sensor_map)
std  = df[sensors].std()
nuni = df[sensors].nunique()

keep = [s for s in sensors if std[s] >= 1e-6 and nuni[s] >= 3]
EXP4_KEEP = ["s2","s3","s4","s7","s8","s9","s11","s12","s13","s14","s15","s17","s20","s21"]

assert keep == EXP4_KEEP, "Sensor set differs from Experiment 4!"
print("Kept", len(keep), "sensors:", keep)

Kept 14 sensors: ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']


### Why covariance-based PCA would mislead

In [6]:
X = df[keep].to_numpy(dtype=float, copy=True)
n, p = X.shape

var = pd.Series(X.var(axis=0, ddof=1), index=keep)
share = (var / var.sum() * 100).sort_values(ascending=False)

print(f"n = {n}, p = {p}")
print("Share of TOTAL raw variance (%):")
print(share.round(2))
print(f"\ns9 + s14 together: {share[['s9','s14']].sum():.1f}% of total variance")

n = 20631, p = 14
Share of TOTAL raw variance (%):
s9     50.05
s14    37.35
s4      8.32
s3      3.86
s17     0.25
s7      0.08
s12     0.06
s2      0.03
s11     0.01
s20     0.00
s21     0.00
s13     0.00
s8      0.00
s15     0.00
dtype: float64

s9 + s14 together: 87.4% of total variance


#### Teaching point: 

Look at how much of the total raw variance comes from s9 and s14 together. If that share is large, covariance-based PCA would make PC1 essentially "core speed", and Experiment 4 showed those are the weakest RUL trackers (|r| = 0.46 and 0.37). That's a units effect, not physics.

### Standardise by hand and check against scikit-learn

In [7]:
mu    = X.mean(axis=0)
sigma = X.std(axis=0, ddof=1)          # sample std, as in Exp 4
Z     = (X - mu) / sigma

Z_sk = StandardScaler().fit_transform(X)   # uses ddof = 0 (population std)

ratio = Z / Z_sk
print("Z / Z_sklearn  min, max :", ratio.min().round(8), ratio.max().round(8))
print("sqrt((n-1)/n)           :", round(np.sqrt((n - 1) / n), 8))

Z / Z_sklearn  min, max : 0.99997576 0.99997576
sqrt((n-1)/n)           : 0.99997576


#### Teaching point: 

StandardScaler divides by n, not n−1, so the two versions differ by a constant factor sqrt{(n-1)/n} = 0.99998. Scaling every column by the same constant doesn't change any eigenvector, so the principal directions are identical. Only the eigenvalues are scaled by (n−1)/n.

### Check that cov(Z) = R

In [8]:
R_from_Z = (Z.T @ Z) / (n - 1)
R_pandas = df[keep].corr().to_numpy()

print("Column means ~0   :", np.allclose(Z.mean(axis=0), 0))
print("Column stds = 1   :", np.allclose(Z.std(axis=0, ddof=1), 1))
print("max |Z'Z/(n-1) - R|:", np.abs(R_from_Z - R_pandas).max())
print("trace             :", np.trace(R_from_Z).round(6), "(= p)")

Column means ~0   : True
Column stds = 1   : True
max |Z'Z/(n-1) - R|: 9.285822111237962e-12
trace             : 14.0 (= p)


### Save the clean data and Z for later steps

In [9]:
clean = df[["unit", "cycle", "RUL", "RUL_cap"] + keep].copy()
clean.to_csv(PROC_DIR / "fd001_clean.csv", index=False)

np.save(PROC_DIR / "Z.npy", Z)
pd.Series(mu, index=keep).to_csv(PROC_DIR / "mu.csv")
pd.Series(sigma, index=keep).to_csv(PROC_DIR / "sigma.csv")
print("Saved:", clean.shape, Z.shape)

Saved: (20631, 18) (20631, 14)
